In [ ]:
import math, time, warnings
from contextlib import contextmanager
from dataclasses import dataclass
from typing import Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda import amp

warnings.filterwarnings("ignore")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {props.total_memory/1e9:.1f} GB")
    print(f"SMs    : {props.multi_processor_count}")
    print(f"CUDA   : {props.major}.{props.minor}")


def benchmark(fn, warmup=20, iters=200):
    """Generic CUDA benchmark. fn is a zero-argument callable."""
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(iters):
        fn()
    torch.cuda.synchronize()
    return (time.perf_counter() - t0) / iters * 1e3  # ms

---
# 1) GPU vs CPU: SIMT Architecture & Warp Execution

A modern CPU optimises for **latency** (branch prediction, out-of-order execution, large caches).
A GPU optimises for **throughput** via massive parallelism.

**SIMT — Single Instruction, Multiple Threads:**
Groups of 32 threads called *warps* execute the **same instruction** simultaneously on different data.
This maps perfectly to dense linear algebra where every element undergoes the same floating-point op.

**Warp divergence:**
If threads in a warp take different code paths (`if/else`), the GPU serialises both paths,
masking inactive threads. Every branch costs every thread. This is why ML kernels are branch-free.

| Concept | CPU | GPU |
|---|---|---|
| Parallelism unit | Core (1 thread) | Warp (32 threads) |
| Latency hiding | Branch prediction | Thread-context switching |
| Best suited for | Sequential logic | Dense linear algebra |

**Sample — warp occupancy:**
- SM with max 2048 threads: `2048 / 32 = 64` resident warps
- Block of 256 threads = 8 warps → 8 blocks per SM at full occupancy


In [ ]:
def warp_occupancy_analysis():
    if not torch.cuda.is_available():
        print("CUDA not available.")
        return
    props = torch.cuda.get_device_properties(0)
    warp_size = 32
    max_warps_sm = props.max_threads_per_multi_processor // warp_size
    max_blocks_sm = props.max_blocks_per_multi_processor

    print("=" * 55)
    print("  SM-Level Warp Occupancy Analysis")
    print("=" * 55)
    print(f"  Warp size                    : {warp_size}")
    print(f"  Max resident warps / SM      : {max_warps_sm}")
    print(f"  Max blocks / SM              : {max_blocks_sm}")
    print(f"  Streaming Multiprocessors    : {props.multi_processor_count}")
    print(
        f"  Total resident threads       : {max_warps_sm*warp_size*props.multi_processor_count:,}"
    )
    print()
    print(
        f"  {'Block size':>12} | {'Warps/block':>11} | {'Blocks/SM':>9} | {'Occupancy':>10}"
    )
    print(f"  {'-'*50}")
    for bs in [32, 64, 128, 256, 512, 1024]:
        wpb = bs // warp_size
        bsm = min(max_blocks_sm, max_warps_sm // wpb)
        occ = bsm * wpb / max_warps_sm
        print(f"  {bs:>12} | {wpb:>11} | {bsm:>9} | {occ:>9.1%}")


warp_occupancy_analysis()

In [ ]:
# Empirical warp-divergence cost: branchy vs branch-free elementwise op


def branchy(x: torch.Tensor) -> torch.Tensor:
    # Odd/even elements take different paths — intra-warp divergence
    return torch.where(x % 2 == 0, x * 2.0, x * 3.0 + 1.0)


def branchless(x: torch.Tensor) -> torch.Tensor:
    # Algebraically equivalent, no branch
    factor = 2.0 + (x % 2) * 1.0
    offset = (x % 2) * 1.0
    return x * factor + offset


if torch.cuda.is_available():
    # (num_samples,)
    x = torch.arange(1, 2**24 + 1, dtype=torch.float32, device=DEVICE)
    assert torch.allclose(branchy(x), branchless(x)), "Mismatch"
    t_b = benchmark(lambda: branchy(x))
    t_bl = benchmark(lambda: branchless(x))
    print(f"Branchy    : {t_b:.3f} ms")
    print(f"Branchless : {t_bl:.3f} ms")
    print(
        f"Speedup    : {t_b/t_bl:.2f}x  (PyTorch fuses these; divergence penalty is more"
    )
    print("             pronounced in raw CUDA/Triton kernels)")

---
# 2) CUDA Programming Model: Grids, Blocks, Threads

## Intuition & Motivation

CUDA organises threads in a three-level hierarchy:
```
Grid  (launch config)
 └── Blocks  (share L1/shared memory; can __syncthreads())
      └── Threads  (each executes the kernel body)
```
Each thread identifies itself via `blockIdx` and `threadIdx`:
```c
int row = blockIdx.y * blockDim.y + threadIdx.y;
int col = blockIdx.x * blockDim.x + threadIdx.x;
```

**Why blocks?** Threads in the same block share ~96 KB of programmer-managed SRAM
(called shared memory) and can synchronise cheaply. Cross-block communication requires global memory.

**Sample — 2-D matrix addition (1024×1024):**
```
Block shape : (16, 16) → 256 threads / block
Grid  shape : (64, 64) → 4096 blocks
Total threads: 1,048,576 — one per element
```

**Guard condition:** When the matrix size is not divisible by the block size,
the last block contains out-of-bounds threads that must be masked:
```c
if (row < M && col < N) C[row*N+col] = A[row*N+col] + B[row*N+col];
```


In [ ]:
def show_launch_config(rows: int, cols: int, bx: int, by: int) -> None:
    """Print CUDA launch configuration and thread→element mapping."""
    gx = math.ceil(cols / bx)
    gy = math.ceil(rows / by)
    total = gx * gy * bx * by
    useful = rows * cols
    print(f"Matrix ({rows},{cols})  block ({by},{bx})  grid ({gy},{gx})")
    print(
        f"  Total threads : {total:,}   Useful: {useful:,}  Efficiency: {useful/total:.1%}"
    )
    print(f"  Waste         : {total-useful:,}  (guard condition required)")
    print()


show_launch_config(1024, 1024, 16, 16)
show_launch_config(1000, 1000, 16, 16)  # non-divisible

# Canonical CUDA add kernel (C equivalent shown as docstring)
CUDA_ADD_KERNEL = """
// CUDA kernel — 1 thread per element
__global__ void add_kernel(const float* a, const float* b, float* c, int N) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < N) c[idx] = a[idx] + b[idx];   // guard condition
}
// Launch: add_kernel<<<ceil(N/256), 256>>>(a, b, c, N);
"""
print("CUDA kernel (reference):")
print(CUDA_ADD_KERNEL)

# PyTorch equivalent — identical semantics, dispatches to CUDA
N = 1 << 24
a = torch.randn(N, device=DEVICE)
b = torch.randn(N, device=DEVICE)

# (num_samples,) → (num_samples,)
c = a + b

t = benchmark(lambda: a + b)
bw = N * 3 * 4 / t * 1e-9 * 1e3  # read a, read b, write c
print(f"Element-wise add ({N/1e6:.0f}M floats): {t:.3f} ms  |  {bw:.1f} GB/s")

---
# 3) GPU Memory Hierarchy

## Intuition & Motivation

| Level | Size | Latency | BW | Programmer control |
|---|---|---|---|---|
| **Registers** | ~256 KB / SM | 1 cycle | ~20 TB/s | Implicit (compiler) |
| **L1 / Shared** | 32–128 KB / SM | ~30 cycles | ~10 TB/s | `__shared__` |
| **L2 cache** | 4–40 MB | ~200 cycles | ~2–4 TB/s | Hardware |
| **Global (HBM)** | 16–80 GB | ~600 cycles | ~900 GB/s | `cudaMalloc` |

**Tiled matmul:** Naïve matmul issues O(N³) HBM reads. Tiling loads a (tile×tile) block into
shared memory, reducing HBM traffic by `tile_size`. FlashAttention is the attention equivalent.

**Coalescing rule:** 32 consecutive threads accessing 32 consecutive 4-byte addresses
→ 1 × 128-byte transaction. Stride-2 access → 2 transactions. Random → 32 transactions.


In [ ]:
# Demonstrate coalesced vs strided HBM access


def strided_access(x: torch.Tensor, stride: int) -> torch.Tensor:
    # (num_samples,) → (num_samples//stride,)
    return x[::stride].clone()


def coalesced_access(x: torch.Tensor) -> torch.Tensor:
    # (num_samples,) → (num_samples,)
    return x.clone()


if torch.cuda.is_available():
    x = torch.randn(2**26, device=DEVICE)
    print(f"{'Pattern':>16} | {'ms':>8} | {'GB/s':>8}")
    print("-" * 38)
    t = benchmark(lambda: coalesced_access(x))
    bw = 2 * x.numel() * 4 / t * 1e-9 * 1e3
    print(f"{'Coalesced':>16} | {t:>8.3f} | {bw:>8.1f}")
    for stride in [2, 4, 8, 16, 32]:
        t2 = benchmark(lambda s=stride: strided_access(x, s), warmup=5, iters=50)
        bw2 = x[::stride].numel() * 4 / t2 * 1e-9 * 1e3
        print(f"{'Stride-'+str(stride):>16} | {t2:>8.3f} | {bw2:>8.1f}")

In [ ]:
# Tiled matmul: demonstrate shared-memory tiling algorithm in Python


def matmul_tiled(
    A: torch.Tensor,  # (M, K)
    B: torch.Tensor,  # (K, N)
    tile: int = 64,
) -> torch.Tensor:
    """
    Mirrors the CUDA shared-memory tiled matmul algorithm.
    Each iteration loads one (M, tile) slab of A and (tile, N) slab of B
    into 'shared memory' (a Python local variable here), accumulating C in registers.
    Real CUDA: __shared__ float A_smem[tile][tile]; __syncthreads();
    """
    M, K = A.shape
    _, N = B.shape
    # Accumulator in FP32 even for FP16 inputs (mirrors Tensor Core behaviour)
    # (M, N)
    C = torch.zeros(M, N, device=A.device, dtype=torch.float32)
    for k0 in range(0, K, tile):
        # Load tile into 'shared memory'
        # (M, tile_k) and (tile_k, N)
        At = A[:, k0 : k0 + tile].float()
        Bt = B[k0 : k0 + tile, :].float()
        # Accumulate partial dot products
        # (M, tile_k) × (tile_k, N) → (M, N)
        C += At @ Bt
    return C


M, K, N = 256, 128, 256
A = torch.randn(M, K, device=DEVICE)
B = torch.randn(K, N, device=DEVICE)
err = (A @ B - matmul_tiled(A, B)).abs().max().item()
print(f"Tiled vs reference max error: {err:.2e}")

# Benchmark matmul across sizes (cuBLAS/cuTLASS in FP16)
print(f"{'Size':>12} | {'ms':>8} | {'TFLOPS':>8}")
print("-" * 36)
for s in [256, 512, 1024, 2048, 4096]:
    X = torch.randn(s, s, device=DEVICE, dtype=torch.float16)
    Y = torch.randn(s, s, device=DEVICE, dtype=torch.float16)
    t = benchmark(lambda: torch.mm(X, Y))
    tfl = 2 * s**3 / t / 1e9
    print(f"{s}x{s}x{s:>5} | {t:>8.3f} | {tfl:>8.2f}")

---
# 4) Memory Bandwidth vs Compute Bound: The Roofline Model

## Intuition & Motivation

The **Roofline model** classifies every kernel as either bandwidth- or compute-limited.

```
Arithmetic Intensity (AI) = FLOP / bytes_from_HBM
Attainable GFLOP/s        = min(AI × peak_BW,  peak_FLOPS)
```

The **ridge point** (AI where the two roofs intersect) = `peak_FLOPS / peak_BW`.
- AI < ridge → **memory bound** → fuse ops, increase reuse
- AI > ridge → **compute bound** → use lower precision, algorithmic improvements

| Operation | AI (FP32) | Bound |
|---|---|---|
| Vector add | 0.08 FLOP/byte | Memory |
| Layer Norm | ~0.5 | Memory |
| Softmax | ~1.0 | Memory |
| Large GEMM | 32–256 | Compute |
| Flash Attention | ~2 | Memory |


In [ ]:
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np


def plot_roofline(
    peak_bw: float, peak_tflops: float, kernels: dict, title: str
) -> None:
    peak_gflops = peak_tflops * 1e3
    ai_ridge = peak_gflops / peak_bw
    ai_x = np.logspace(-2, 4, 500)
    roof = np.minimum(peak_bw * ai_x, peak_gflops)

    fig, ax = plt.subplots(figsize=(10, 5.5))
    ax.set_facecolor("#0d1117")
    fig.patch.set_facecolor("#0d1117")
    ax.loglog(ai_x, roof, color="#58a6ff", lw=2.5, label="Roofline")
    ax.axvline(ai_ridge, color="#f0883e", ls="--", lw=1.2, alpha=0.8)
    ax.text(
        ai_ridge * 1.1,
        peak_gflops * 0.45,
        f"Ridge {ai_ridge:.0f} FLOP/B",
        color="#f0883e",
        fontsize=9,
    )
    ax.fill_between(
        ai_x,
        roof,
        peak_gflops,
        where=ai_x < ai_ridge,
        alpha=0.07,
        color="#3fb950",
        label="Memory-bound",
    )
    ax.fill_between(
        ai_x,
        roof,
        peak_gflops,
        where=ai_x >= ai_ridge,
        alpha=0.07,
        color="#f85149",
        label="Compute-bound",
    )
    cols = ["#ffa657", "#79c0ff", "#56d364", "#ff7b72", "#d2a8ff", "#e3b341"]
    for (name, (ai, perf)), col in zip(kernels.items(), cols):
        ax.scatter(ai, perf, color=col, s=80, zorder=5)
        ax.annotate(
            name,
            (ai, perf),
            xytext=(8, 4),
            textcoords="offset points",
            color=col,
            fontsize=8.5,
        )
    ax.set(
        xlabel="Arithmetic Intensity (FLOP/byte)",
        ylabel="Performance (GFLOP/s)",
        title=title,
        xlim=(1e-2, 1e4),
        ylim=(1, peak_gflops * 2),
    )
    for sp in ax.spines.values():
        sp.set_edgecolor("#30363d")
    ax.tick_params(colors="#c9d1d9")
    ax.grid(color="#21262d", lw=0.5)
    ax.set_xlabel("Arithmetic Intensity (FLOP/byte)", color="#c9d1d9")
    ax.set_ylabel("Performance (GFLOP/s)", color="#c9d1d9")
    ax.set_title(title, color="#c9d1d9")
    ax.legend(
        facecolor="#161b22", edgecolor="#30363d", labelcolor="#c9d1d9", fontsize=9
    )
    plt.tight_layout()
    plt.savefig("/tmp/roofline.png", dpi=140, bbox_inches="tight")
    plt.show()


# T4: ~320 GB/s BW, ~65 TFLOPS FP16
BW, TFLOPS = 320.0, 65.0
kernels = {
    "Vector Add": (0.08, BW * 0.08 * 0.85),
    "Layer Norm": (0.5, BW * 0.5 * 0.80),
    "Softmax": (1.0, BW * 1.0 * 0.75),
    "Flash Attention": (2.0, BW * 2.0 * 0.70),
    "GEMM 512": (32, TFLOPS * 1e3 * 0.55),
    "GEMM 4096": (256, TFLOPS * 1e3 * 0.85),
}
plot_roofline(BW, TFLOPS, kernels, "Roofline — NVIDIA T4 (FP16 Tensor Core)")

In [ ]:
# Measure actual HBM bandwidth on the current GPU via a large D2D memcpy


def measure_hbm_bandwidth(size_gb: float = 0.5) -> Tuple[float, float]:
    num_floats = int(size_gb * 1e9) // 4
    x = torch.randn(num_floats, device=DEVICE)
    y = torch.empty_like(x)
    for _ in range(5):
        y.copy_(x)
    torch.cuda.synchronize()
    t = benchmark(lambda: y.copy_(x), warmup=5, iters=30)
    bw = 2 * num_floats * 4 / t * 1e-9 * 1e3  # read + write
    return bw, t


if torch.cuda.is_available():
    bw, t = measure_hbm_bandwidth(0.5)
    print(f"HBM bandwidth (0.5 GB copy): {bw:.1f} GB/s  ({t:.2f} ms/iter)")
    N = 2**24
    a = torch.randn(N, device=DEVICE)
    b = torch.randn(N, device=DEVICE)
    t_add = benchmark(lambda: a + b)
    ai = 1 / 12  # 1 FLOP / 12 bytes (read a+b, write c, FP32)
    print(f"Vector add AI               : {ai:.3f} FLOP/byte → memory-bound")
    print(f"Achieved GFLOP/s            : {N/t_add/1e6:.1f}")

---
# 5) Tensor Cores & Mixed-Precision Matrix Multiply

## Intuition & Motivation

**Tensor Cores** execute a 4×4×4 matrix-multiply-accumulate in a single clock cycle:
```
D[4×4] = A[4×4] × B[4×4] + C[4×4]
```
This gives **4–8× FLOP throughput** over FP32 CUDA cores.

| GPU | FP32 FLOPS | FP16 TC | BF16 TC | FP8 TC |
|---|---|---|---|---|
| T4 | 8 TFLOPS | 65 TFLOPS | — | — |
| A100 | 19 TFLOPS | 312 TFLOPS | 312 TFLOPS | — |
| H100 | 67 TFLOPS | 989 TFLOPS | 989 TFLOPS | 1979 TFLOPS |

**AMP (Automatic Mixed Precision) recipe:**
1. Keep master weights in FP32
2. Cast to FP16/BF16 for forward + backward (Tensor Core path)
3. Scale loss by `S` to prevent FP16 gradient underflow
4. Unscale gradients, check for inf/nan, update FP32 master weights

**Sample — FP16 matmul speedup:** 4096×4096 matmul: FP32 ≈ 6 TFLOPS vs FP16 Tensor Core ≈ 50 TFLOPS → ~8× speedup.


In [ ]:
# Benchmark FP32 vs FP16 vs BF16 matmul


def dtype_benchmark(M: int, K: int, N: int) -> None:
    A32 = torch.randn(M, K, device=DEVICE)
    B32 = torch.randn(K, N, device=DEVICE)
    C32 = torch.mm(A32, B32)
    flops = 2 * M * K * N
    print(f"Shape ({M},{K})×({K},{N})")
    print(f"{'Dtype':>6} | {'ms':>8} | {'TFLOPS':>8} | {'Max err':>10}")
    print("-" * 42)
    for name, dtype in [
        ("FP32", torch.float32),
        ("FP16", torch.float16),
        ("BF16", torch.bfloat16),
    ]:
        A = A32.to(dtype)
        B = B32.to(dtype)
        C = torch.mm(A, B)
        t = benchmark(lambda: torch.mm(A, B))
        err = (C.float() - C32).abs().max().item()
        print(f"{name:>6} | {t:>8.3f} | {flops/t/1e9:>8.2f} | {err:>10.4f}")


dtype_benchmark(4096, 4096, 4096)

In [ ]:
# Full AMP training loop with GradScaler


class TinyBlock(nn.Module):
    def __init__(self, model_dim: int, num_heads: int):
        super().__init__()
        self.attn = nn.MultiheadAttention(model_dim, num_heads, batch_first=True)
        self.ff = nn.Sequential(
            nn.Linear(model_dim, model_dim * 4),
            nn.GELU(),
            nn.Linear(model_dim * 4, model_dim),
        )
        self.ln1 = nn.LayerNorm(model_dim)
        self.ln2 = nn.LayerNorm(model_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # (batch_num, seq_len, model_dim) → (batch_num, seq_len, model_dim)
        x = x + self.attn(self.ln1(x), self.ln1(x), self.ln1(x))[0]
        return x + self.ff(self.ln2(x))


def amp_demo(
    steps: int = 40,
    batch_num: int = 8,
    seq_len: int = 64,
    model_dim: int = 256,
    num_heads: int = 4,
) -> None:
    model = TinyBlock(model_dim, num_heads).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
    scaler = amp.GradScaler(enabled=torch.cuda.is_available())
    print(f"{'Step':>5} | {'Loss':>8} | {'Scale':>10}")
    print("-" * 32)
    for step in range(steps):
        # (batch_num, seq_len, model_dim)
        x = torch.randn(batch_num, seq_len, model_dim, device=DEVICE)
        tgt = torch.randn_like(x)
        opt.zero_grad(set_to_none=True)
        with amp.autocast(device_type="cuda", enabled=torch.cuda.is_available()):
            # (batch_num, seq_len, model_dim) → scalar
            loss = F.mse_loss(model(x), tgt)
        # Scale loss → backward → unscale → step → update scale
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()
        if step % 10 == 0:
            print(f"{step:>5} | {loss.item():>8.4f} | {scaler.get_scale():>10.1f}")


amp_demo()

---
# 6) CUDA Kernels for ML: MatMul, Attention, Layer Norm

## Intuition & Motivation

Custom kernels exist for three reasons:
1. **Fusion** — merging ops eliminates intermediate HBM round-trips.
2. **Algorithm changes** — FlashAttention rewrites attention to avoid materialising the O(seq_len²) score matrix.
3. **Layout exploitation** — cuBLAS/cuTLASS choose tiles; custom kernels exploit problem structure.

### Flash Attention Core Idea
Standard attention: write full (seq_len × seq_len) scores to HBM → O(seq_len²) memory.
FlashAttention: compute partial softmax on-chip with online normalisation:
```
for each tile of Q:
    load tile of K, V into SRAM
    compute partial scores + online softmax update (m, l)
    accumulate output O in SRAM
write O to HBM once
```
HBM traffic: **O(seq_len × head_dim)** instead of O(seq_len²).

### Online Softmax (Welford-style, numerically stable)
```
m_new = max(m_old, tile_max)
l_new = exp(m_old − m_new)·l_old + sum(exp(x − m_new))
O     = diag(exp(m_old−m_new))·O_old + exp(x−m_new)·V
```


In [ ]:
# Standard scaled dot-product attention — explicit O(seq_len²) score matrix


def attention_standard(
    Q: "torch.Tensor",  # (batch_num, num_heads, seq_len, head_dim)
    K: "torch.Tensor",  # (batch_num, num_heads, seq_len, head_dim)
    V: "torch.Tensor",  # (batch_num, num_heads, seq_len, head_dim)
    mask: "Optional[torch.Tensor]" = None,
) -> "torch.Tensor":
    import math

    scale = math.sqrt(Q.size(-1))
    # (batch_num, num_heads, seq_len, seq_len)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / scale
    if mask is not None:
        scores = scores + mask
    # (batch_num, num_heads, seq_len, head_dim)
    return torch.matmul(torch.softmax(scores, dim=-1), V)


def make_causal_mask(seq_len: int, device) -> "torch.Tensor":
    # (1, 1, seq_len, seq_len)
    return (
        torch.triu(
            torch.full((seq_len, seq_len), float("-inf"), device=device), diagonal=1
        )
        .unsqueeze(0)
        .unsqueeze(0)
    )


# Validate standard vs PyTorch FlashAttention (F.scaled_dot_product_attention)
import torch, torch.nn.functional as F

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def benchmark(fn, warmup=20, iters=200):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    import time

    t0 = time.perf_counter()
    for _ in range(iters):
        fn()
    torch.cuda.synchronize()
    return (time.perf_counter() - t0) / iters * 1e3


batch_num, num_heads, seq_len, head_dim = 2, 8, 256, 64
Q = torch.randn(
    batch_num, num_heads, seq_len, head_dim, device=DEVICE, dtype=torch.float16
)
K, V = torch.randn_like(Q), torch.randn_like(Q)
mask = make_causal_mask(seq_len, DEVICE).half()

out_std = attention_standard(Q, K, V, mask)
with torch.backends.cuda.sdp_kernel(
    enable_flash=True, enable_math=False, enable_mem_efficient=True
):
    out_flash = F.scaled_dot_product_attention(Q, K, V, attn_mask=mask)

print(f"Max deviation (std vs flash): {(out_std-out_flash).abs().max().item():.4e}")
print()
print(f"{'seq_len':>8} | {'std (ms)':>10} | {'flash (ms)':>10} | {'speedup':>8}")
print("-" * 44)
for s in [128, 256, 512, 1024]:
    Q2 = torch.randn(2, 8, s, 64, device=DEVICE, dtype=torch.float16)
    K2, V2, m2 = (
        torch.randn_like(Q2),
        torch.randn_like(Q2),
        make_causal_mask(s, DEVICE).half(),
    )
    ts = benchmark(lambda: attention_standard(Q2, K2, V2, m2), warmup=10, iters=50)
    with torch.backends.cuda.sdp_kernel(
        enable_flash=True, enable_math=False, enable_mem_efficient=True
    ):
        tf = benchmark(
            lambda: F.scaled_dot_product_attention(Q2, K2, V2, attn_mask=m2),
            warmup=10,
            iters=50,
        )
    print(f"{s:>8} | {ts:>10.2f} | {tf:>10.2f} | {ts/tf:>8.2f}x")

In [ ]:
# Layer Norm from scratch — two-pass and Welford (single-pass) variants


def layer_norm_twopass(
    x: "torch.Tensor",  # (batch_num, seq_len, model_dim)
    gamma: "torch.Tensor",  # (model_dim,)
    beta: "torch.Tensor",  # (model_dim,)
    eps: float = 1e-5,
) -> "torch.Tensor":
    """Standard two-pass layer norm (mean, then variance)."""
    # (batch_num, seq_len, 1)
    mu = x.mean(dim=-1, keepdim=True)
    var = x.var(dim=-1, keepdim=True, unbiased=False)
    # (batch_num, seq_len, model_dim)
    return gamma * (x - mu) / (var + eps).sqrt() + beta


def layer_norm_welford(
    x: "torch.Tensor",  # (batch_num, seq_len, model_dim)
    gamma: "torch.Tensor",
    beta: "torch.Tensor",
    eps: float = 1e-5,
) -> "torch.Tensor":
    """
    Welford online mean+variance — single pass, numerically stable.
    Avoids catastrophic cancellation when subtracting a large mean.
    Real GPU kernels use this to stay within a single SRAM tile.
    """
    D = x.shape[-1]
    mean = torch.zeros(*x.shape[:-1], 1, device=x.device, dtype=x.dtype)
    M2 = torch.zeros_like(mean)
    for i in range(D):
        n = i + 1
        # (batch_num, seq_len, 1)
        delta = x[..., i : i + 1] - mean
        mean = mean + delta / n
        M2 = M2 + delta * (x[..., i : i + 1] - mean)
    var = M2 / D
    return gamma * (x - mean) / (var + eps).sqrt() + beta


import torch, torch.nn as nn

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def benchmark(fn, warmup=20, iters=200):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    import time

    t0 = time.perf_counter()
    for _ in range(iters):
        fn()
    torch.cuda.synchronize()
    return (time.perf_counter() - t0) / iters * 1e3


batch_num, seq_len, model_dim = 4, 32, 128
x = torch.randn(batch_num, seq_len, model_dim, device=DEVICE)
gamma = torch.ones(model_dim, device=DEVICE)
beta = torch.zeros(model_dim, device=DEVICE)
ref = nn.LayerNorm(model_dim).to(DEVICE)
ref.weight.data.fill_(1.0)
ref.bias.data.fill_(0.0)

out_ref = ref(x)
out_tp = layer_norm_twopass(x, gamma, beta)
out_welf = layer_norm_welford(x, gamma, beta)
print(f"Two-pass vs ref  : {(out_ref-out_tp).abs().max().item():.2e}")
print(f"Welford vs ref   : {(out_ref-out_welf).abs().max().item():.2e}")

x_large = torch.randn(64, 512, 768, device=DEVICE)
g, b = torch.ones(768, device=DEVICE), torch.zeros(768, device=DEVICE)
ref_l = nn.LayerNorm(768).to(DEVICE)
t_ref = benchmark(lambda: ref_l(x_large), warmup=10, iters=100)
t_tp = benchmark(lambda: layer_norm_twopass(x_large, g, b), warmup=10, iters=100)
print(f"\nLayerNorm (64x512x768): cuDNN={t_ref:.2f}ms  manual={t_tp:.2f}ms")

---
# 7) Triton: Writing Custom GPU Kernels in Python

## Intuition & Motivation

**Triton** (OpenAI 2021) compiles Python-like kernel code to PTX/SASS, sitting between
PyTorch and raw CUDA C++. You think in *blocks of elements*, not individual threads.

```python
@triton.jit
def kernel(X, Y, Z, N, BLOCK: tl.constexpr):
    pid  = tl.program_id(0)              # ≡ blockIdx.x
    offs = pid * BLOCK + tl.arange(0, BLOCK)   # ≡ threadIdx.x range
    mask = offs < N
    tl.store(Z + offs, tl.load(X+offs,mask=mask) + tl.load(Y+offs,mask=mask), mask=mask)
```

Triton auto-generates coalesced loads/stores, vectorisation, and warp-synchronisation.

### Fused Softmax — 1 HBM pass instead of 3
Naïve: subtract_max → exp → divide_by_sum  (3 full passes over the row)
Fused: load row once → subtract max in registers → exp → sum → divide → store


In [ ]:
try:
    import triton
    import triton.language as tl

    TRITON_OK = True
    print(f"Triton {triton.__version__} available")
except ImportError:
    TRITON_OK = False
    print("Triton not installed — install with: pip install triton")

In [ ]:
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def benchmark(fn, warmup=20, iters=200):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    import time

    t0 = time.perf_counter()
    for _ in range(iters):
        fn()
    torch.cuda.synchronize()
    return (time.perf_counter() - t0) / iters * 1e3


# Triton vector add kernel
if TRITON_OK and torch.cuda.is_available():

    @triton.jit
    def triton_add_kernel(X, Y, Z, N, BLOCK: tl.constexpr):
        """
        One Triton program processes BLOCK contiguous elements.
        tl.program_id(0) ≡ blockIdx.x
        tl.arange(0,BLOCK) spans the thread range within this block
        """
        pid = tl.program_id(0)
        offs = pid * BLOCK + tl.arange(0, BLOCK)
        mask = offs < N
        # Coalesced 128-byte loads, arithmetic in registers, single store
        tl.store(
            Z + offs,
            tl.load(X + offs, mask=mask, other=0.0)
            + tl.load(Y + offs, mask=mask, other=0.0),
            mask=mask,
        )

    def triton_add(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        # (num_samples,) → (num_samples,)
        N = x.numel()
        z = torch.empty_like(x)
        BLOCK = 1024
        triton_add_kernel[(triton.cdiv(N, BLOCK),)](x, y, z, N, BLOCK_SIZE=BLOCK)
        return z

    N = 1 << 24
    x = torch.randn(N, device=DEVICE)
    y = torch.randn(N, device=DEVICE)
    assert torch.allclose(triton_add(x, y), x + y), "Mismatch"
    t_pt = benchmark(lambda: x + y)
    t_tr = benchmark(lambda: triton_add(x, y))
    print(f"PyTorch add : {t_pt:.3f} ms")
    print(f"Triton add  : {t_tr:.3f} ms")
    print("(Overhead visible on simple ops; Triton shines on fused kernels)")

In [ ]:
import torch, torch.nn.functional as F

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def benchmark(fn, warmup=20, iters=200):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    import time

    t0 = time.perf_counter()
    for _ in range(iters):
        fn()
    torch.cuda.synchronize()
    return (time.perf_counter() - t0) / iters * 1e3


# Triton fused softmax — single HBM pass per row
if TRITON_OK and torch.cuda.is_available():

    @triton.jit
    def fused_softmax_kernel(X, Y, num_cols, BLOCK: tl.constexpr):
        """
        One program per row. Entire row fits in registers (BLOCK ≥ num_cols).
        Three softmax passes happen without HBM round-trips:
          1. Load row → registers
          2. Subtract max (numerical stability)
          3. exp + sum → normalise
          4. Store output
        """
        row = tl.program_id(0)
        base = row * num_cols
        cols = tl.arange(0, BLOCK)
        mask = cols < num_cols
        # Single HBM load
        x = tl.load(X + base + cols, mask=mask, other=float("-inf"))
        # All three passes in registers
        x = x - tl.max(x, axis=0)
        x = tl.exp(x)
        y = x / tl.sum(x, axis=0)
        # Single HBM store
        tl.store(Y + base + cols, y, mask=mask)

    def triton_softmax(x: torch.Tensor) -> torch.Tensor:
        # (num_rows, num_cols) → (num_rows, num_cols)
        num_rows, num_cols = x.shape
        BLOCK = triton.next_power_of_2(num_cols)
        y = torch.empty_like(x)
        fused_softmax_kernel[(num_rows,)](x, y, num_cols, BLOCK=BLOCK)
        return y

    R, C = 1024, 4096
    x = torch.randn(R, C, device=DEVICE)
    assert torch.allclose(
        triton_softmax(x), F.softmax(x, dim=-1), atol=1e-5
    ), "Mismatch"
    t_pt = benchmark(lambda: F.softmax(x, dim=-1))
    t_tr = benchmark(lambda: triton_softmax(x))
    bw_pt = 2 * R * C * 4 / t_pt * 1e-9 * 1e3
    bw_tr = 2 * R * C * 4 / t_tr * 1e-9 * 1e3
    print(f"PyTorch softmax: {t_pt:.3f} ms  {bw_pt:.1f} GB/s")
    print(f"Triton  softmax: {t_tr:.3f} ms  {bw_tr:.1f} GB/s")
    print(f"Speedup        : {t_pt/t_tr:.2f}x")

---
# 8) Profiling: torch.profiler & NVIDIA Nsight

## Intuition & Motivation

**torch.profiler** wraps CUPTI (CUDA Performance Tools Interface) to record kernel durations,
memory allocations, and CUDA streams. Output can be exported as a Chrome trace or TensorBoard event.

**Workflow:**
1. `torch.profiler.profile(activities=[CPU, CUDA], ...)` context manager
2. `prof.key_averages()` → table of top kernels
3. `prof.export_chrome_trace("trace.json")` → open in `chrome://tracing`
4. For cycle-accurate kernel metrics → **Nsight Compute** (`ncu --metrics ...`)

**What to look for:**
- Long `cudaLaunchKernel` in CPU trace → small kernel launches (overhead-bound)
- `cudaMemcpy` in hot path → H2D/D2H copies inside training loop
- Gaps between CUDA kernels → CPU–GPU synchronisation (`loss.item()` in loop!)
- Memory timeline → activation memory reuse opportunities


In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import torch.profiler as profiler
from torch.cuda import amp

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def make_causal_mask(seq_len, device):
    return (
        torch.triu(
            torch.full((seq_len, seq_len), float("-inf"), device=device), diagonal=1
        )
        .unsqueeze(0)
        .unsqueeze(0)
    )


def attention_standard(Q, K, V, mask=None):
    import math

    s = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(Q.size(-1))
    if mask is not None:
        s = s + mask
    return torch.matmul(torch.softmax(s, dim=-1), V)


def profile_attention(batch_num=4, num_heads=8, seq_len=512, head_dim=64):
    if not torch.cuda.is_available():
        print("CUDA not available.")
        return
    Q = torch.randn(
        batch_num, num_heads, seq_len, head_dim, device=DEVICE, dtype=torch.float16
    )
    K, V = torch.randn_like(Q), torch.randn_like(Q)
    mask = make_causal_mask(seq_len, DEVICE).half()

    # Warm-up both variants
    for _ in range(5):
        attention_standard(Q, K, V, mask)
        with torch.backends.cuda.sdp_kernel(
            enable_flash=True, enable_math=False, enable_mem_efficient=True
        ):
            F.scaled_dot_product_attention(Q, K, V, attn_mask=mask)
    torch.cuda.synchronize()

    with profiler.profile(
        activities=[profiler.ProfilerActivity.CPU, profiler.ProfilerActivity.CUDA],
        record_shapes=True,
        profile_memory=True,
    ) as prof:
        with profiler.record_function("standard_attention"):
            attention_standard(Q, K, V, mask)
        torch.cuda.synchronize()
        with profiler.record_function("flash_attention"):
            with torch.backends.cuda.sdp_kernel(
                enable_flash=True, enable_math=False, enable_mem_efficient=True
            ):
                F.scaled_dot_product_attention(Q, K, V, attn_mask=mask)
        torch.cuda.synchronize()

    print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=8))
    prof.export_chrome_trace("/tmp/attn_trace.json")
    print("Chrome trace → /tmp/attn_trace.json  (open in chrome://tracing)")


profile_attention()

In [ ]:
import torch, torch.nn as nn
from torch.cuda import amp

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# Memory profiling: track peak VRAM at each phase of a training step
def profile_memory(batch_num=16, seq_len=256, model_dim=512, num_heads=8, num_layers=4):
    if not torch.cuda.is_available():
        print("CUDA not available.")
        return
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.empty_cache()

    model = nn.TransformerEncoder(
        nn.TransformerEncoderLayer(
            d_model=model_dim,
            nhead=num_heads,
            dim_feedforward=model_dim * 4,
            batch_first=True,
            dtype=torch.float16,
        ),
        num_layers=num_layers,
    ).to(DEVICE)

    mb = lambda: torch.cuda.memory_allocated() / 1e6
    pmb = lambda: torch.cuda.max_memory_allocated() / 1e6
    torch.cuda.reset_peak_memory_stats()

    print(f"After model load    : {mb():7.1f} MB  (peak {pmb():7.1f})")
    x = torch.randn(batch_num, seq_len, model_dim, device=DEVICE, dtype=torch.float16)
    print(f"After input alloc   : {mb():7.1f} MB  (peak {pmb():7.1f})")

    # Forward stores activations for backward → large memory spike
    with amp.autocast(device_type="cuda"):
        out = model(x)
    torch.cuda.synchronize()
    print(f"After forward       : {mb():7.1f} MB  (peak {pmb():7.1f})")

    out.mean().backward()
    torch.cuda.synchronize()
    print(f"After backward      : {mb():7.1f} MB  (peak {pmb():7.1f})")

    grad_mb = (
        sum(
            p.grad.element_size() * p.grad.numel()
            for p in model.parameters()
            if p.grad is not None
        )
        / 1e6
    )
    print(f"Gradient memory     : {grad_mb:7.1f} MB")
    print(f"Peak VRAM (step)    : {pmb():7.1f} MB")


profile_memory()

---
# 9) Data Parallelism: DDP & Gradient All-Reduce

## Intuition & Motivation

**Data parallelism** replicates the full model on N GPUs, splits the batch N ways,
then synchronises gradients via all-reduce before the optimizer step.

```
GPU i:  batch[i*B/N : (i+1)*B/N]  →  local_grad_i
All-Reduce: global_grad = Σ local_grad_i / N
All GPUs:   optimizer.step(global_grad)
```

**DDP vs DataParallel:**
| | DataParallel | DDP |
|---|---|---|
| Topology | GPU 0 gathers all grads | Ring all-reduce (peer-to-peer) |
| Bottleneck | GPU 0 VRAM + PCIe | Balanced |
| Process model | Single process | Multi-process (no GIL) |

**Gradient bucketing:** DDP groups parameters into 25 MB buckets.
All-reduce fires as a bucket fills during backward, overlapping comms with the next layer's backward.

**Ring all-reduce = Reduce-Scatter + All-Gather** in two phases, each O(N-1) messages.


In [ ]:
import torch, math


def ring_all_reduce(tensors):
    """
    Simulate ring all-reduce across world_size virtual GPUs.
    Phase 1 (Reduce-Scatter): GPU i accumulates the global sum for shard i.
    Phase 2 (All-Gather):     each GPU broadcasts its summed shard to all others.
    Returns: list of tensors, each holding the global sum.
    """
    world_size = len(tensors)
    N = tensors[0].numel()
    chunk = math.ceil(N / world_size)
    bufs = [t.clone() for t in tensors]

    # Phase 1: Reduce-Scatter
    for step in range(world_size - 1):
        for rank in range(world_size):
            send_i = (rank - step) % world_size
            recv_i = (rank - step - 1) % world_size
            dst = (rank + 1) % world_size
            s0, s1 = send_i * chunk, min((send_i + 1) * chunk, N)
            r0, r1 = recv_i * chunk, min((recv_i + 1) * chunk, N)
            bufs[dst][r0:r1] += bufs[rank][s0:s1]

    # Phase 2: All-Gather
    for step in range(world_size - 1):
        for rank in range(world_size):
            send_i = (rank - step + 1) % world_size
            recv_i = (rank - step) % world_size
            dst = (rank + 1) % world_size
            s0, s1 = send_i * chunk, min((send_i + 1) * chunk, N)
            r0, r1 = recv_i * chunk, min((recv_i + 1) * chunk, N)
            bufs[dst][r0:r1] = bufs[rank][s0:s1].clone()
    return bufs


# Validate: ring all-reduce must equal simple sum
world_size, N = 4, 17  # intentionally non-divisible
grads = [torch.randn(N) for _ in range(world_size)]
expected = sum(grads)
result = ring_all_reduce(grads)
for r, res in enumerate(result):
    err = (res - expected).abs().max().item()
    print(f"Rank {r}: max error = {err:.2e}  {'✓' if err<1e-5 else '✗'}")

In [ ]:
# DDP training loop pattern (annotated pseudocode)
print("""# ── torchrun --nproc_per_node=8 train.py ────────────────────────────────────
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

dist.init_process_group(backend="nccl")
rank, world = dist.get_rank(), dist.get_world_size()
torch.cuda.set_device(rank)

model    = MyModel().cuda(rank)
ddp_model = DDP(model, device_ids=[rank])  # registers all-reduce hooks
# ↑ On backward: each bucket of gradients triggers a ring all-reduce automatically

sampler = DistributedSampler(dataset, num_replicas=world, rank=rank)
loader  = DataLoader(dataset, sampler=sampler, batch_size=64)
opt     = torch.optim.AdamW(ddp_model.parameters(), lr=1e-4)

for epoch in range(N):
    sampler.set_epoch(epoch)   # different shuffle each epoch
    for x, y in loader:
        opt.zero_grad()
        loss = criterion(ddp_model(x.cuda(rank)), y.cuda(rank))
        loss.backward()
        # DDP all-reduces gradients here (async, overlaps with remaining backward)
        opt.step()

dist.destroy_process_group()
""")

---
# 10) Model Parallelism: Tensor & Pipeline Parallelism

## Intuition & Motivation

When weights don't fit on one GPU, we shard the *model* itself.

### Tensor Parallelism (Megatron-LM style)
Split weight matrices **column-wise or row-wise** across GPUs.

**Column-parallel (no comm in forward):**
```
Y = X @ W,  W: (in, out)
Split W into [W0|W1|...|W_{N-1}] along columns
GPU i: Y_i = X @ W_i   ← independent, no sync
Concat: Y = cat(Y_i)
```

**Row-parallel (all-reduce after):**
```
Each GPU holds a row shard: input is split, outputs are summed.
GPU i: out_i = X_i @ W_i  →  all-reduce(sum(out_i)) = Y
```

**MLP: 1 all-reduce per layer** (column then row; all-reduce after row).

### Pipeline Parallelism (1F1B schedule)
Layers split across GPUs (stage). Micro-batches flow through the pipeline.

**Bubble fraction (1F1B):** `(p−1)/(m+p−1)`  where p=stages, m=micro-batches.
Rule of thumb: m ≥ 4p keeps bubble < 20%.


In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class ColumnParallelLinear(nn.Module):
    """Splits output dimension across tp_size GPUs. No communication in forward."""

    def __init__(self, in_features, out_features, tp_size):
        super().__init__()
        assert out_features % tp_size == 0
        self.local_out = out_features // tp_size
        # tp_size × (in_features, local_out)
        self.shards = nn.ParameterList(
            [
                nn.Parameter(torch.randn(in_features, self.local_out) * 0.02)
                for _ in range(tp_size)
            ]
        )

    def forward(self, x):
        # (batch_num, ..., in_features) → (batch_num, ..., out_features)
        return torch.cat([x @ w for w in self.shards], dim=-1)


class RowParallelLinear(nn.Module):
    """Splits input dimension. Requires all-reduce (simulated by sum) after."""

    def __init__(self, in_features, out_features, tp_size):
        super().__init__()
        assert in_features % tp_size == 0
        self.local_in = in_features // tp_size
        self.tp_size = tp_size
        # tp_size × (local_in, out_features)
        self.shards = nn.ParameterList(
            [
                nn.Parameter(torch.randn(self.local_in, out_features) * 0.02)
                for _ in range(tp_size)
            ]
        )

    def forward(self, x):
        # (batch_num, ..., in_features) → (batch_num, ..., out_features)
        x_parts = x.split(self.local_in, dim=-1)
        return sum(xi @ w for xi, w in zip(x_parts, self.shards))


class TensorParallelMLP(nn.Module):
    """Megatron-style MLP: column-linear → GELU → row-linear (1 all-reduce/layer)."""

    def __init__(self, model_dim, ff_dim, tp_size):
        super().__init__()
        self.col = ColumnParallelLinear(model_dim, ff_dim, tp_size)
        self.row = RowParallelLinear(ff_dim, model_dim, tp_size)

    def forward(self, x):
        # (batch_num, seq_len, model_dim) → (batch_num, seq_len, model_dim)
        return self.row(F.gelu(self.col(x)))


# Validate TP MLP against standard MLP (same weights)
batch_num, seq_len, model_dim, ff_dim, tp_size = 2, 16, 64, 128, 4
tp_mlp = TensorParallelMLP(model_dim, ff_dim, tp_size).to(DEVICE)
ref_mlp = nn.Sequential(
    nn.Linear(model_dim, ff_dim, bias=False),
    nn.GELU(),
    nn.Linear(ff_dim, model_dim, bias=False),
).to(DEVICE)

with torch.no_grad():
    ref_mlp[0].weight.data = torch.cat([w.T for w in tp_mlp.col.shards], dim=0)
    ref_mlp[2].weight.data = torch.cat([w.T for w in tp_mlp.row.shards], dim=1)

x = torch.randn(batch_num, seq_len, model_dim, device=DEVICE)
with torch.no_grad():
    out_ref = ref_mlp(x)
    out_tp = tp_mlp(x)

print(f"TP vs reference max error: {(out_tp-out_ref).abs().max().item():.4e}")
print(f"Communication: 1 all-reduce per forward pass (at row-parallel output)")

In [ ]:
import math


def pipeline_bubble_table(stages_list, microbatch_list):
    """
    Bubble fraction for 1F1B schedule:  (p-1) / (m + p - 1)
    p = pipeline stages, m = micro-batches per optimizer step.
    """
    print("1F1B Pipeline Bubble Fraction")
    header = "m \\ p"
    print(f"{header:>6}" + "".join(f"{p:>8}" for p in stages_list))
    print("-" * (6 + 8 * len(stages_list)))
    for m in microbatch_list:
        row = f"{m:>6}"
        for p in stages_list:
            row += f"{(p-1)/(m+p-1):>8.1%}"
        print(row)
    print("Rule of thumb: m ≥ 4p keeps bubble < 20%")


pipeline_bubble_table([2, 4, 8, 16], [4, 8, 16, 32, 64])

---
# 11) ZeRO Optimization (Stages 1, 2, 3) — DeepSpeed

## Intuition & Motivation

Training memory per parameter (FP16 training with Adam):
| State | bytes/param |
|---|---|
| Parameters (FP16) | 2 |
| Gradients (FP16) | 2 |
| Adam m + v (FP32) | 8 |
| Master weights (FP32) | 4 |
| **Total** | **16** |

A 7B model needs 112 GB → impossible on one 80 GB GPU.

**ZeRO shards these states:**
| Stage | Sharded | Memory / GPU |
|---|---|---|
| Baseline | Nothing | 16 × P |
| ZeRO-1 | Optimizer states | ≈ (2+2)P + 12P/N |
| ZeRO-2 | + Gradients | ≈ 2P + 14P/N |
| ZeRO-3 | + Parameters | 16P/N |

**Communication overhead:** ZeRO-3 adds 1 extra all-gather per layer (reconstruct params),
but eliminates the gradient all-reduce → same asymptotic communication as DDP for large models.


In [ ]:
from dataclasses import dataclass


@dataclass
class ZeROProfile:
    world_size: int
    num_params: float  # in billions
    p_bytes: int = 2  # FP16 param/grad bytes
    opt_bytes: int = 12  # FP32: 4 master + 4 m + 4 v

    def baseline(self):
        return self.num_params * 1e9 * (self.p_bytes * 2 + self.opt_bytes)

    def zero1(self):
        return (
            self.num_params
            * 1e9
            * (self.p_bytes * 2 + self.opt_bytes / self.world_size)
        )

    def zero2(self):
        return (
            self.num_params
            * 1e9
            * (self.p_bytes + (self.p_bytes + self.opt_bytes) / self.world_size)
        )

    def zero3(self):
        return (
            self.num_params
            * 1e9
            * (self.p_bytes * 2 + self.opt_bytes)
            / self.world_size
        )


def print_zero_table(param_b, world_sizes):
    print(f"Memory/GPU (GB) — {param_b}B params")
    print(f"{'GPUs':>6}|{'Baseline':>10}|{'ZeRO-1':>10}|{'ZeRO-2':>10}|{'ZeRO-3':>10}")
    print("-" * 50)
    for ws in world_sizes:
        p = ZeROProfile(ws, param_b)
        print(
            f"{ws:>6}|{p.baseline()/1e9:>10.1f}|{p.zero1()/1e9:>10.1f}|{p.zero2()/1e9:>10.1f}|{p.zero3()/1e9:>10.1f}"
        )


print_zero_table(7, [1, 8, 64, 512])
print()
print_zero_table(70, [8, 64, 512, 1024])

In [ ]:
import torch, torch.nn as nn, math


class ZeROStage1Adam:
    """
    ZeRO Stage 1: optimizer state (m, v, master weights) is sharded.
    Rank i owns the optimizer state for parameters where (param_idx % world_size == i).
    After each step, updated parameters are broadcast via simulated all-gather.
    """

    def __init__(self, params, world_size, rank, lr=1e-4, betas=(0.9, 0.999), eps=1e-8):
        self.params, self.world_size, self.rank = params, world_size, rank
        self.lr, self.betas, self.eps = lr, betas, eps
        self.t = 0
        # Only allocate state for owned parameters (1/world_size of total)
        self.m = {
            i: torch.zeros_like(p, dtype=torch.float32)
            for i, p in enumerate(params)
            if i % world_size == rank
        }
        self.v = {
            i: torch.zeros_like(p, dtype=torch.float32)
            for i, p in enumerate(params)
            if i % world_size == rank
        }

    def step(self, grads):
        self.t += 1
        b1, b2 = self.betas
        bc1, bc2 = 1 - b1**self.t, 1 - b2**self.t
        updated = [p.clone() for p in self.params]
        for i, (p, g) in enumerate(zip(self.params, grads)):
            if i % self.world_size == self.rank:
                g32 = g.float()
                self.m[i] = b1 * self.m[i] + (1 - b1) * g32
                self.v[i] = b2 * self.v[i] + (1 - b2) * g32**2
                mh = self.m[i] / bc1
                vh = self.v[i] / bc2
                updated[i] = (p.float() - self.lr * mh / (vh.sqrt() + self.eps)).to(
                    p.dtype
                )
        # Simulated all-gather: owning rank broadcasts its updated param
        for i in range(len(self.params)):
            self.params[i].copy_(updated[i])


# Validate ZeRO-1 Adam vs standard Adam
torch.manual_seed(42)
plist = [torch.randn(16, 16) for _ in range(8)]
std_p = [p.clone().requires_grad_(True) for p in plist]
grads = [torch.randn_like(p) for p in plist]
for p, g in zip(std_p, grads):
    p.grad = g.clone()
std_opt = torch.optim.Adam(std_p, lr=1e-4)
std_opt.step()

z1p = [p.clone() for p in plist]
z1gs = [g.clone() for g in grads]
for r in range(4):
    ZeROStage1Adam(z1p, world_size=4, rank=r, lr=1e-4).step(z1gs)

errs = [(sp.data - zp).abs().max().item() for sp, zp in zip(std_p, z1p)]
print("ZeRO-1 Adam vs standard Adam max errors:")
for i, e in enumerate(errs):
    print(f"  param {i}: {e:.4e} {'✓' if e<1e-5 else '✗'}")

# 12) Fully Sharded Data Parallelism (FSDP)

## Motivation
ZeRO-3 shards **everything** (params, grads, optimizer states) across GPUs but
lives inside DeepSpeed. PyTorch's **FSDP** brings the same idea natively, with
tight integration into `autograd` and `torch.distributed`.

### Key insight — "all-gather before forward, re-shard after"
```
GPU 0 holds shard p[0]   GPU 1 holds shard p[1]   GPU 2 holds shard p[2]
        ↓ all-gather before forward pass
 Each GPU briefly holds the full weight W = [p[0] | p[1] | p[2]]
        ↓ compute forward / backward as normal
 Each GPU keeps only its gradient shard; free the full W immediately
```

### FSDP vs DDP memory
| Strategy | Params | Grads | Opt states | Total (fp32, Adam) |
|----------|--------|-------|------------|--------------------|
| DDP      | Ψ      | Ψ     | 8Ψ         | 10Ψ bytes          |
| FSDP     | Ψ/N   | Ψ/N  | 8Ψ/N      | 10Ψ/N bytes        |

FSDP peak memory ≈ Ψ/N + transient Ψ during forward all-gather.

### Sample
```python
model = MyTransformer()
fsdp_model = FSDP(model, auto_wrap_policy=transformer_auto_wrap_policy)
fsdp_model(x).sum().backward()  # sharding/unsharding handled automatically
```


In [ ]:
import math
from dataclasses import dataclass, field
from typing import Dict, List

import torch
import torch.nn as nn


# ── Minimal MiniGPT used throughout FSDP examples ────────────────────────────
class CausalSelfAttention(nn.Module):
    """Standard multi-head causal self-attention."""

    def __init__(self, embed_dim: int, num_heads: int) -> None:
        super().__init__()
        assert embed_dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.qkv = nn.Linear(embed_dim, 3 * embed_dim, bias=False)
        self.proj = nn.Linear(embed_dim, embed_dim, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # (batch_num, seq_len, embed_dim) → (batch_num, seq_len, embed_dim)
        B, T, C = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.num_heads, self.head_dim)
        q, k, v = qkv.unbind(2)
        q, k, v = (t.transpose(1, 2) for t in (q, k, v))
        # (batch_num, num_heads, seq_len, head_dim) → same
        y = torch.nn.functional.scaled_dot_product_attention(q, k, v, is_causal=True)
        y = y.transpose(1, 2).reshape(B, T, C)
        return self.proj(y)


class TransformerBlock(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int) -> None:
        super().__init__()
        self.ln1 = nn.LayerNorm(embed_dim)
        self.attn = CausalSelfAttention(embed_dim, num_heads)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, 4 * embed_dim),
            nn.GELU(),
            nn.Linear(4 * embed_dim, embed_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # (batch_num, seq_len, embed_dim) → (batch_num, seq_len, embed_dim)
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class MiniGPT(nn.Module):
    """Small GPT-style model for sharding demonstrations."""

    def __init__(
        self,
        vocab_size: int = 50_257,
        embed_dim: int = 256,
        num_heads: int = 4,
        num_layers: int = 4,
        max_len: int = 256,
    ) -> None:
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.pos = nn.Embedding(max_len, embed_dim)
        self.blocks = nn.ModuleList(
            [TransformerBlock(embed_dim, num_heads) for _ in range(num_layers)]
        )
        self.ln_f = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, vocab_size, bias=False)

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        # (batch_num, seq_len) → (batch_num, seq_len, vocab_size)
        B, T = idx.shape
        tok = self.embed(idx)
        pos = self.pos(torch.arange(T, device=idx.device).unsqueeze(0))
        x = tok + pos
        for block in self.blocks:
            x = block(x)
        return self.head(self.ln_f(x))


model = MiniGPT()
total_params = sum(p.numel() for p in model.parameters())
print(f"MiniGPT  params: {total_params:,}  ({total_params * 4 / 1e6:.1f} MB fp32)")

## 12.1 FSDP Memory Simulation

We can't run `torch.distributed.fsdp.FSDP` without a multi-GPU cluster, but
we **can** simulate the memory math precisely and show what FSDP wraps.


In [ ]:
@dataclass
class FSDPMemoryProfile:
    """Byte-accurate memory breakdown per FSDP sharding strategy."""

    model_name: str
    total_params: int
    num_gpus: int
    param_bytes: int = 2  # bf16 / fp16 for forward
    grad_bytes: int = 2  # bf16 / fp16
    master_param_bytes: int = 4  # fp32 master copy (mixed precision)
    opt_state_bytes: int = 8  # Adam: m + v in fp32

    def _shard(self, bytes_per_param: int, sharded: bool) -> float:
        """Return GB for a given per-param cost, optionally sharded."""
        total = self.total_params * bytes_per_param
        return total / (self.num_gpus if sharded else 1) / 1e9

    def report(self) -> None:
        strategies = {
            "DDP (no sharding)": dict(params=False, grads=False, optimizer=False),
            "ZeRO-1 / FSDP opt-shard": dict(params=False, grads=False, optimizer=True),
            "ZeRO-2 / FSDP grad+opt-shard": dict(
                params=False, grads=True, optimizer=True
            ),
            "ZeRO-3 / FSDP full-shard": dict(params=True, grads=True, optimizer=True),
        }
        print(f"\n{'Model':12s} {self.model_name}")
        print(f"{'Params':12s} {self.total_params:,}")
        print(f"{'GPUs':12s} {self.num_gpus}")
        print(
            f"\n{'Strategy':<35} {'Params':>9} {'Grads':>9} "
            f"{'Opt':>9} {'Total':>9} GB / GPU"
        )
        print("-" * 75)
        for name, flags in strategies.items():
            p = self._shard(self.param_bytes + self.master_param_bytes, flags["params"])
            g = self._shard(self.grad_bytes, flags["grads"])
            o = self._shard(self.opt_state_bytes, flags["optimizer"])
            total = p + g + o
            print(f"{name:<35} {p:>9.3f} {g:>9.3f} {o:>9.3f} {total:>9.3f}")


# Demonstrate for MiniGPT and a GPT-3-scale model
for name, n_params in [("MiniGPT-256", total_params), ("GPT-3-175B", 175_000_000_000)]:
    FSDPMemoryProfile(name, n_params, num_gpus=8).report()
    print()

## 12.2 FSDP Pseudocode — Forward / Backward Pass

The pseudocode below makes every all-gather and reduce-scatter explicit,
showing exactly **when** tensors are materialised and freed.


In [ ]:
def fsdp_forward_pseudocode(num_layers: int = 4) -> None:
    """
    Walk through an FSDP forward + backward pass layer by layer,
    printing each communication and memory event.
    """
    comm_vol_per_layer_GB = total_params / num_layers * 2 / 1e9  # bf16

    print("═" * 60)
    print("FSDP FORWARD PASS")
    print("═" * 60)
    for i in range(num_layers):
        print(
            f"  Layer {i}: all_gather(shard_W{i})  "
            f"→ materialise full W{i}  [{comm_vol_per_layer_GB:.3f} GB]"
        )
        print(f"         : forward(x, W{i})  →  activations saved")
        print(f"         : free W{i}  ← shard back to W{i}[local]")

    print()
    print("═" * 60)
    print("FSDP BACKWARD PASS")
    print("═" * 60)
    for i in reversed(range(num_layers)):
        print(
            f"  Layer {i}: all_gather(shard_W{i})  "
            f"→ re-materialise W{i}  [{comm_vol_per_layer_GB:.3f} GB]"
        )
        print(f"         : backward(grad_out, W{i})  →  grad_W{i} (full)")
        print(f"         : reduce_scatter(grad_W{i}) →  keep local grad shard")
        print(f"         : free W{i}, free full grad_W{i}")

    total_comm = 2 * num_layers * comm_vol_per_layer_GB  # fwd + bwd
    print(f"\nTotal all-gather + reduce-scatter volume: {total_comm:.3f} GB")
    print(
        "(DDP equivalent: 1 all-reduce = 2× params bytes = "
        f"{total_params * 2 * 2 / 1e9:.3f} GB)"
    )


fsdp_forward_pseudocode()

# 13) Communication Primitives

## Motivation
Distributed training requires moving data between GPUs via well-defined
**collective operations**. Understanding their semantics and volume is essential
for diagnosing bottlenecks.

| Primitive        | Input per GPU | Output per GPU | Volume (total) |
|------------------|---------------|----------------|----------------|
| Broadcast        | M (root only) | M              | (N-1)M         |
| Reduce           | M             | M (root only)  | (N-1)M         |
| All-Reduce       | M             | M              | 2(N-1)M/N ≈ 2M |
| All-Gather       | M/N           | M              | (N-1)M/N       |
| Reduce-Scatter   | M             | M/N            | (N-1)M/N       |

Note: All-Reduce = Reduce-Scatter + All-Gather (ring algorithm).

### Sample
```
4 GPUs, each holds [a, b, c, d]:
All-Reduce  → each GPU holds [a+b+c+d, ...] (sum)
All-Gather  → each GPU holds full [a|b|c|d]
Reduce-Scat → GPU 0 holds sum of col 0, GPU 1 of col 1, ...
```


In [ ]:
import torch


def simulate_broadcast(
    tensors: list[torch.Tensor], root: int = 0
) -> list[torch.Tensor]:
    """
    Broadcast: root GPU sends its tensor to all others.
    Volume: (N-1) * M bytes
    """
    # (num_gpus, *shape) → each GPU receives root's tensor
    result = [tensors[root].clone() for _ in tensors]
    return result


def simulate_reduce(tensors: list[torch.Tensor], root: int = 0) -> list[torch.Tensor]:
    """
    Reduce: all GPUs send; root accumulates the sum.
    Non-root GPUs end up with their original (unchanged) tensors.
    """
    # (num_gpus, *shape) → root gets element-wise sum
    result = [t.clone() for t in tensors]
    result[root] = torch.stack(tensors).sum(dim=0)
    return result


def simulate_all_reduce(tensors: list[torch.Tensor]) -> list[torch.Tensor]:
    """
    All-Reduce: every GPU ends up with the element-wise sum.
    Equivalent to ring reduce-scatter followed by ring all-gather.
    """
    # (num_gpus, *shape) → (num_gpus, *shape) all equal to sum
    total = torch.stack(tensors).sum(dim=0)
    return [total.clone() for _ in tensors]


def simulate_all_gather(shards: list[torch.Tensor]) -> list[torch.Tensor]:
    """
    All-Gather: each GPU contributes its shard; all GPUs receive the full tensor.
    Input shard shape: (shard_size,)  Output: (N * shard_size,)
    """
    # (num_gpus, shard_size) → (num_gpus, full_size)
    full = torch.cat(shards, dim=0)
    return [full.clone() for _ in shards]


def simulate_reduce_scatter(tensors: list[torch.Tensor]) -> list[torch.Tensor]:
    """
    Reduce-Scatter: sum across GPUs, then scatter one chunk to each GPU.
    Input: (num_gpus, full_size)   Output: (num_gpus, shard_size)
    """
    # (num_gpus, full_size) → (num_gpus, shard_size)
    N = len(tensors)
    total = torch.stack(tensors).sum(dim=0)  # (full_size,)
    shard_size = total.shape[0] // N
    return [total[i * shard_size : (i + 1) * shard_size].clone() for i in range(N)]


# ── Verification ─────────────────────────────────────────────────────────────
N, M = 4, 8
data = [torch.arange(float(M)) * (i + 1) for i in range(N)]

print("Input tensors (one per GPU):")
for i, t in enumerate(data):
    print(f"  GPU {i}: {t.tolist()}")

print("\nAll-Reduce (should be element-wise sum):")
ar = simulate_all_reduce(data)
print(f"  GPU 0: {ar[0].tolist()}")

shards = [torch.tensor([float(i)]) for i in range(N)]  # shard_size=1
print("\nAll-Gather input shards:", [s.item() for s in shards])
full = simulate_all_gather(shards)
print(f"  Full tensor (each GPU): {full[0].tolist()}")

print("\nReduce-Scatter output (one chunk per GPU):")
rs = simulate_reduce_scatter(data)
for i, s in enumerate(rs):
    print(f"  GPU {i}: {s.tolist()}")

## 13.1 Ring All-Reduce — Step-by-Step

The **ring** algorithm achieves bandwidth-optimal all-reduce in two phases:
1. **Reduce-Scatter** (N-1 steps): each GPU sends its chunk to the right; after
   N-1 steps every GPU holds one fully-reduced chunk.
2. **All-Gather** (N-1 steps): forward those reduced chunks around the ring.

Total bytes sent per GPU = **2(N-1)/N × M** ≈ 2M (bandwidth-optimal).


In [ ]:
def ring_all_reduce_verbose(chunks: list[torch.Tensor]) -> list[torch.Tensor]:
    """
    Bandwidth-optimal ring all-reduce with step-by-step logging.
    Each GPU starts with a chunk of the full tensor.

    Args:
        chunks: list of length N, each tensor shape (chunk_size,)

    Returns:
        List of N tensors, each equal to the element-wise sum of all inputs.
    """
    N = len(chunks)
    # Each GPU holds the full tensor split into N chunks
    # buffers[gpu][chunk_idx]
    buffers: list[list[torch.Tensor]] = [[c.clone() for c in chunks] for _ in range(N)]

    print(f"Ring All-Reduce  N={N} GPUs, {N} chunks")
    print("─" * 50)

    # Phase 1: Reduce-Scatter
    print("PHASE 1 — Reduce-Scatter")
    for step in range(N - 1):
        send_chunk = (torch.arange(N) - step) % N  # which chunk each GPU sends
        for gpu in range(N):
            src = (gpu - 1) % N
            chunk_idx = int(send_chunk[src].item())
            buffers[gpu][chunk_idx] += buffers[src][chunk_idx]
        print(
            f"  step {step+1}: GPU i accumulates chunk {int(send_chunk[0].item())} "
            f"from GPU (i-1)%{N}"
        )

    # Phase 2: All-Gather
    print("PHASE 2 — All-Gather")
    for step in range(N - 1):
        send_chunk = (torch.arange(N) - step - 1) % N
        for gpu in range(N):
            src = (gpu - 1) % N
            chunk_idx = int(send_chunk[src].item())
            buffers[gpu][chunk_idx] = buffers[src][chunk_idx].clone()
        print(
            f"  step {step+1}: GPU i copies reduced chunk {int(send_chunk[0].item())} "
            f"from GPU (i-1)%{N}"
        )

    # Reconstruct full tensors
    result = [torch.cat(buf) for buf in buffers]
    return result


chunks_in = [torch.ones(4) * (i + 1) for i in range(4)]
print("Input (per GPU):", [c.tolist() for c in chunks_in])
result = ring_all_reduce_verbose(chunks_in)
print("\nOutput (per GPU, first GPU shown):", result[0].tolist())
expected = sum(i + 1 for i in range(4)) * torch.ones(4)
assert torch.allclose(result[0], expected), "Ring all-reduce mismatch!"
print("✓ Result matches naive all-reduce")

# 14) NCCL, NVLink, and Multi-Node Networking

## Motivation
The **communication topology** between GPUs determines the achievable
collective bandwidth and therefore whether training is compute- or
communication-bound.

### Hardware hierarchy
```
Node (server)
├── PCIe root complex
│   ├── GPU 0 ─── NVLink ──── GPU 1        (600 GB/s bidirectional NVLink 4)
│   ├── GPU 2 ─── NVLink ──── GPU 3
│   └── NVSwitch ────────────────────────── (connects all 8 GPUs @ 900 GB/s)
└── NIC (InfiniBand / RoCE)
        ↕ RDMA (400 Gb/s HDR200)
Node 2 NIC
```

### NCCL topology selection
NCCL auto-detects the fastest path:
- **Same GPU**: shared memory
- **Same node, NVLink**: use NVSwitch fabric
- **Same node, PCIe**: use CPU memory as relay
- **Cross-node**: RDMA via `ib_verbs` or RoCE (no CPU copy)

### Key metric: *bus bandwidth* vs *algorithmic bandwidth*
Ring all-reduce theoretical peak = `link_BW × (N-1)/N`


In [ ]:
from dataclasses import dataclass, field
from typing import Literal


@dataclass
class Interconnect:
    """Model a GPU interconnect topology and predict all-reduce time."""

    name: str
    link_bw_GBps: float  # unidirectional bandwidth GB/s
    num_gpus: int
    latency_us: float = 1.0  # baseline per-hop latency

    @property
    def algo_bw_GBps(self) -> float:
        """Ring all-reduce algorithmic bandwidth = link_bw * (N-1)/N."""
        N = self.num_gpus
        return self.link_bw_GBps * (N - 1) / N

    def all_reduce_time_ms(self, tensor_bytes: int) -> float:
        """
        Estimate all-reduce latency:
          time = 2(N-1)/N * bytes / link_bw  + 2(N-1) * latency
        """
        N = self.num_gpus
        data_ms = 2 * (N - 1) / N * tensor_bytes / (self.link_bw_GBps * 1e9) * 1e3
        lat_ms = 2 * (N - 1) * self.latency_us * 1e-3
        return data_ms + lat_ms


# ── Typical configurations ────────────────────────────────────────────────────
configs = [
    Interconnect("PCIe Gen4", 32, 8, latency_us=5.0),
    Interconnect("NVLink 3 (A100)", 600, 8, latency_us=1.5),
    Interconnect("NVLink 4 (H100)", 900, 8, latency_us=1.2),
    Interconnect("InfiniBand HDR", 25, 64, latency_us=2.0),  # 200Gb/s ≈ 25 GB/s
    Interconnect("InfiniBand NDR", 50, 64, latency_us=1.5),  # 400Gb/s
]

# Gradient tensor sizes for a 7B model: params × 2 bytes (bf16)
gradient_bytes = 7_000_000_000 * 2

print(
    f"{'Interconnect':<25} {'Gpus':>5} {'Link BW':>10} {'Algo BW':>10}"
    f" {'AR time (ms)':>14}"
)
print("─" * 70)
for ic in configs:
    t_ms = ic.all_reduce_time_ms(gradient_bytes)
    print(
        f"{ic.name:<25} {ic.num_gpus:>5} {ic.link_bw_GBps:>9.0f} "
        f"{ic.algo_bw_GBps:>9.1f}  {t_ms:>13.1f}"
    )

print(f"\nGradient tensor (7B model, bf16): {gradient_bytes/1e9:.1f} GB")

## 14.1 Communication-Compute Overlap

Modern frameworks pipeline **backward computation** with gradient communication
so that all-reduce of layer *i*'s gradient overlaps with backward pass of layer
*i-1*.

```
No overlap:    [bwd L3][bwd L2][bwd L1][all-reduce L3][all-reduce L2][all-reduce L1]
With overlap:  [bwd L3][all-reduce L3]
                       [bwd L2][all-reduce L2]
                              [bwd L1][all-reduce L1]
Total time ≈ max(bwd_time, comm_time) instead of bwd_time + comm_time
```


In [ ]:
def overlap_analysis(
    bwd_time_ms: float,
    ar_time_ms: float,
    num_layers: int,
    overlap_fraction: float = 0.95,
) -> None:
    """
    Compare sequential vs overlapped backward + all-reduce scheduling.

    Args:
        bwd_time_ms:      Total backward pass time (ms)
        ar_time_ms:       Total all-reduce time (ms) — e.g., from Interconnect
        num_layers:       Number of transformer layers
        overlap_fraction: Fraction of comm that can be hidden behind compute
    """
    bwd_per_layer = bwd_time_ms / num_layers
    ar_per_layer = ar_time_ms / num_layers

    sequential_ms = bwd_time_ms + ar_time_ms

    # With overlap: each layer's all-reduce hides behind next layer's backward
    hidden_ms = min(bwd_per_layer, ar_per_layer) * num_layers * overlap_fraction
    overlapped_ms = sequential_ms - hidden_ms

    speedup = sequential_ms / overlapped_ms

    print("Communication-Compute Overlap Analysis")
    print("─" * 45)
    print(f"  Backward time (total)   : {bwd_time_ms:>8.1f} ms")
    print(f"  All-Reduce time (total) : {ar_time_ms:>8.1f} ms")
    print(f"  Layers                  : {num_layers:>8d}")
    print(f"  Per-layer bwd / AR      : {bwd_per_layer:.1f} / {ar_per_layer:.1f} ms")
    print(f"\n  Sequential total        : {sequential_ms:>8.1f} ms")
    print(f"  Overlapped total        : {overlapped_ms:>8.1f} ms")
    print(f"  Speedup from overlap    : {speedup:>8.2f}×")
    regime = "compute-bound" if bwd_per_layer > ar_per_layer else "comm-bound"
    print(f"  Bottleneck              : {regime}")


# 7B model on 8× H100 (NVLink 4), rough timings
h100_ar_ms = configs[2].all_reduce_time_ms(gradient_bytes)
overlap_analysis(
    bwd_time_ms=450.0,
    ar_time_ms=h100_ar_ms,
    num_layers=32,
    overlap_fraction=0.90,
)

# 15) Mixed Precision Training — FP16, BF16, FP8

## Motivation
Using lower-precision arithmetic:
- **2× memory** saving (FP32 → FP16/BF16)
- **2–8× throughput** on Tensor Cores
- Risk: underflow / overflow → need **loss scaling** for FP16

### Floating-point format comparison
| Format | Exponent bits | Mantissa bits | Max value | Min normal |
|--------|--------------|---------------|-----------|------------|
| FP32   | 8            | 23            | ~3.4e38   | ~1.2e-38   |
| FP16   | 5            | 10            | 65504     | ~6.1e-5    |
| BF16   | 8            | 7             | ~3.4e38   | ~1.2e-38   |
| FP8 E4M3| 4           | 3             | 448       | ~1.95e-3   |
| FP8 E5M2| 5           | 2             | 57344     | ~1.5e-5    |

**BF16** has the same exponent range as FP32 → no overflow, but less precision.  
**FP16** can overflow for large logits / loss values → requires loss scaling.  
**FP8** is used for forward pass only (H100+); backward typically stays BF16.


In [ ]:
import struct
import math

import torch
import torch.nn as nn


def fp_format_analysis() -> None:
    """Inspect bit layouts and dynamic ranges of ML floating-point formats."""
    formats = {
        "FP32": (torch.float32, 8, 23),
        "FP16": (torch.float16, 5, 10),
        "BF16": (torch.bfloat16, 8, 7),
    }

    test_values = [1.0, 1e-4, 1e4, 65504.0, 1e-38, 1e38]

    print(f"{'Value':>12} | {'FP32':>14} | {'FP16':>14} | {'BF16':>14}")
    print("─" * 62)
    for v in test_values:
        fp32 = torch.tensor(v, dtype=torch.float32).item()
        fp16 = torch.tensor(v, dtype=torch.float16).item()
        bf16 = torch.tensor(v, dtype=torch.bfloat16).item()

        def fmt(x):
            if math.isinf(x):
                return "  ±inf (OVERFLOW)"
            return f"{x:>14.6g}"

        print(f"{v:>12.4g} | {fmt(fp32)} | {fmt(fp16)} | {fmt(bf16)}")

    print(
        "\nKey: FP16 overflows at >65504; BF16 and FP32 share the same exponent range."
    )


fp_format_analysis()

## 15.1 Loss Scaling — Why and How

FP16 gradients can **underflow to zero** if they fall below ~6e-5.
**Loss scaling** multiplies the loss by a large scalar *S* before backward,
then divides the gradients by *S* before the optimizer step.

```
loss_scaled = loss * S
loss_scaled.backward()        # grads are S× larger → no underflow
grad /= S                     # undo scaling before optimizer
```

Dynamic scaling: double *S* every `growth_interval` steps; halve on overflow.


In [ ]:
class ManualGradScaler:
    """
    Manual implementation of PyTorch's GradScaler logic.

    Tracks a dynamic loss scale and checks for grad overflow (inf/nan)
    after each backward pass.
    """

    def __init__(
        self,
        init_scale: float = 2**16,
        growth_factor: float = 2.0,
        backoff_factor: float = 0.5,
        growth_interval: int = 2000,
    ) -> None:
        self.scale = init_scale
        self.growth_factor = growth_factor
        self.backoff_factor = backoff_factor
        self.growth_interval = growth_interval
        self._steps_since_growth = 0
        self._overflow_count = 0

    def scale_loss(self, loss: torch.Tensor) -> torch.Tensor:
        # (scalar) → (scalar)  multiply loss by current scale
        return loss * self.scale

    def unscale_grads_(self, optimizer: torch.optim.Optimizer) -> None:
        """Divide all gradients in-place by current scale."""
        inv_scale = 1.0 / self.scale
        for group in optimizer.param_groups:
            for p in group["params"]:
                if p.grad is not None:
                    # (any_shape) → (any_shape)
                    p.grad.mul_(inv_scale)

    def _check_overflow(self, optimizer: torch.optim.Optimizer) -> bool:
        """Return True if any gradient is inf or nan."""
        for group in optimizer.param_groups:
            for p in group["params"]:
                if p.grad is not None and not torch.isfinite(p.grad).all():
                    return True
        return False

    def step(self, optimizer: torch.optim.Optimizer) -> bool:
        """
        Unscale grads, check for overflow, conditionally step optimizer.

        Returns True if the step was taken (no overflow).
        """
        self.unscale_grads_(optimizer)

        if self._check_overflow(optimizer):
            self.scale *= self.backoff_factor
            self._steps_since_growth = 0
            self._overflow_count += 1
            optimizer.zero_grad()
            return False

        optimizer.step()
        self._steps_since_growth += 1

        if self._steps_since_growth >= self.growth_interval:
            self.scale *= self.growth_factor
            self._steps_since_growth = 0

        return True

    def zero_grad(self, optimizer: torch.optim.Optimizer) -> None:
        optimizer.zero_grad()

    def state_summary(self) -> str:
        return (
            f"scale={self.scale:.0f}  overflows={self._overflow_count}  "
            f"steps_since_growth={self._steps_since_growth}"
        )


# ── Quick smoke test on CPU ───────────────────────────────────────────────────
torch.manual_seed(42)
model = nn.Linear(32, 32, dtype=torch.float16)
optim = torch.optim.Adam(model.parameters(), lr=1e-3)
scaler = ManualGradScaler(init_scale=128, growth_interval=5)

print("ManualGradScaler training loop (CPU, FP16)\n")
for step in range(12):
    x = torch.randn(8, 32, dtype=torch.float16)
    loss = model(x).pow(2).mean()

    optim.zero_grad()
    scaler.scale_loss(loss).backward()
    taken = scaler.step(optim)

    print(
        f"  step {step+1:2d}  loss={loss.item():.4f}  "
        f"step_taken={taken}  {scaler.state_summary()}"
    )

## 15.2 Simulating FP16 Training Instability

We can inject artificial gradient spikes to see how the scaler responds.


In [ ]:
def simulate_fp16_training(
    spike_steps: list[int] | None = None,
    num_steps: int = 20,
) -> None:
    """
    Simulate mixed-precision training with deliberate gradient overflow events.

    Args:
        spike_steps: step indices where we inject an overflow (inf gradient)
        num_steps:   total training steps
    """
    if spike_steps is None:
        spike_steps = [5, 12]

    model = nn.Linear(16, 16, dtype=torch.float16)
    optim = torch.optim.SGD(model.parameters(), lr=0.01)
    scaler = ManualGradScaler(init_scale=64, growth_interval=4)

    scales, losses = [], []

    for step in range(num_steps):
        x = torch.randn(4, 16, dtype=torch.float16)
        loss = model(x).pow(2).mean()

        optim.zero_grad()
        scaler.scale_loss(loss).backward()

        # Inject overflow at designated steps
        if step in spike_steps:
            for p in model.parameters():
                if p.grad is not None:
                    p.grad[0] = float("inf")

        taken = scaler.step(optim)
        scales.append(scaler.scale)
        losses.append(loss.item() if torch.isfinite(loss) else float("nan"))

        marker = " ⚡OVERFLOW" if not taken else ""
        print(
            f"  step {step+1:2d}  scale={scaler.scale:>8.0f}  "
            f"loss={loss.item():.4f}{marker}"
        )

    print(
        f"\nFinal scale: {scaler.scale:.0f}  "
        f"Total overflows: {scaler._overflow_count}"
    )


simulate_fp16_training()

# 16) Model Scaling Laws & Chinchilla

## Motivation
How should we allocate a **fixed compute budget** between model size *N* and
training tokens *D*?

### Kaplan et al. 2020 (GPT-3 era)
Showed loss scales as a power law in N and D independently:
```
L(N) ≈ A · N^(-α)
L(D) ≈ B · D^(-β)
```
Conclusion at the time: **scale N more than D** (models were undertrained).

### Hoffmann et al. 2022 — "Chinchilla"
Optimal compute budget C = 6ND (one forward + backward ≈ 6 FLOPs per param per token).
Optimal allocation:
```
N_opt ∝ C^0.50    D_opt ∝ C^0.50
```
i.e. **equal scaling of N and D**.  Chinchilla (70B, 1.4T tokens) outperforms
GPT-3 (175B, 300B tokens) at the same compute.

### Loss formula (Chinchilla, equation 4)
```
L(N, D) = E + A/N^α + B/D^β
E ≈ 1.69   A ≈ 406.4   B ≈ 410.7   α ≈ 0.34   β ≈ 0.28
```


In [ ]:
import math
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

# ── Chinchilla constants (Hoffmann et al. 2022, Table 3) ─────────────────────
E = 1.6928
A = 406.4
B = 410.7
ALPHA = 0.3392
BETA = 0.2849


def chinchilla_loss(num_params: float, num_tokens: float) -> float:
    """Predicted cross-entropy loss for given model size and token count."""
    return E + A / (num_params**ALPHA) + B / (num_tokens**BETA)


def chinchilla_optimal(compute_budget_flops: float) -> tuple[float, float]:
    """
    Return (N_opt, D_opt) for a given FLOPs budget.
    Uses C = 6·N·D → D = C / (6·N); minimise L(N, C/(6N)) over N.

    Closed-form solution from Chinchilla paper:
        N_opt = (A·α / (B·β))^(1/(α+β)) · (C/6)^(β/(α+β))
        D_opt = C / (6 · N_opt)
    """
    ratio = (A * ALPHA) / (B * BETA)
    n_opt = ratio ** (1 / (ALPHA + BETA)) * (compute_budget_flops / 6) ** (
        BETA / (ALPHA + BETA)
    )
    d_opt = compute_budget_flops / (6 * n_opt)
    return n_opt, d_opt


# ── Compute-optimal allocation table ─────────────────────────────────────────
def print_zero_table() -> None:
    """Print compute-optimal N and D for a range of FLOPs budgets."""
    print(
        f"{'Compute (FLOPs)':>18} {'N_opt (params)':>18} {'D_opt (tokens)':>18} "
        f"{'N/D ratio':>12} {'Loss':>8}"
    )
    print("─" * 82)
    budgets = [1e18, 1e19, 1e20, 1e21, 1e22, 1e23, 1e24]
    for C in budgets:
        N, D = chinchilla_optimal(C)
        loss = chinchilla_loss(N, D)
        print(f"{C:>18.2e} {N:>18.2e} {D:>18.2e} {N/D:>12.4f} {loss:>8.4f}")


print_zero_table()

In [ ]:
def plot_isoflops_curves() -> None:
    """
    Plot iso-FLOPs curves: for a fixed compute budget,
    show L(N, C/(6N)) as a function of N, marking the optimal point.
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor="#0d1117")
    for ax in axes:
        ax.set_facecolor("#0d1117")
        for spine in ax.spines.values():
            spine.set_edgecolor("#30363d")
        ax.tick_params(colors="#e6edf3")
        ax.xaxis.label.set_color("#e6edf3")
        ax.yaxis.label.set_color("#e6edf3")
        ax.title.set_color("#e6edf3")

    colors = ["#58a6ff", "#3fb950", "#f78166", "#d2a8ff", "#ffa657"]

    # Left: iso-FLOPs loss curves
    budgets = [1e19, 1e20, 1e21, 1e22, 1e23]
    for C, col in zip(budgets, colors):
        N_opt, _ = chinchilla_optimal(C)
        N_range = np.logspace(math.log10(N_opt) - 1.5, math.log10(N_opt) + 1.5, 300)
        D_range = C / (6 * N_range)
        # Only keep points where D > 1e8 (need at least 100M tokens)
        valid = D_range > 1e8
        losses = np.array(
            [chinchilla_loss(n, d) for n, d in zip(N_range[valid], D_range[valid])]
        )
        axes[0].plot(N_range[valid], losses, color=col, label=f"C={C:.0e}")
        axes[0].axvline(N_opt, color=col, ls="--", alpha=0.5)

    axes[0].set_xscale("log")
    axes[0].set_xlabel("Model parameters N")
    axes[0].set_ylabel("Predicted loss L(N, D)")
    axes[0].set_title("Iso-FLOPs curves (dashed = N_opt)")
    axes[0].legend(
        facecolor="#161b22", edgecolor="#30363d", labelcolor="#e6edf3", fontsize=8
    )

    # Right: optimal N and D as a function of compute
    C_range = np.logspace(18, 25, 200)
    N_opts = np.array([chinchilla_optimal(C)[0] for C in C_range])
    D_opts = np.array([chinchilla_optimal(C)[1] for C in C_range])

    axes[1].loglog(C_range, N_opts, color="#58a6ff", label="N_opt (params)")
    axes[1].loglog(C_range, D_opts, color="#3fb950", label="D_opt (tokens)")

    # Mark famous models (approx compute)
    landmarks = {
        "GPT-3 (175B)": (3.1e23, 175e9, "tokens"),
        "Chinchilla (70B)": (5.8e23, 70e9, "params"),
        "LLaMA-2 (70B)": (2.1e24, 70e9, "params"),
    }
    for name, (C_est, size, kind) in landmarks.items():
        col = "#58a6ff" if kind == "params" else "#3fb950"
        axes[1].scatter(C_est, size, color=col, s=80, zorder=5)
        axes[1].annotate(
            name,
            (C_est, size),
            textcoords="offset points",
            xytext=(6, 4),
            fontsize=7,
            color="#e6edf3",
        )

    axes[1].set_xlabel("Compute budget (FLOPs)")
    axes[1].set_ylabel("Optimal size")
    axes[1].set_title("Chinchilla-optimal N and D")
    axes[1].legend(facecolor="#161b22", edgecolor="#30363d", labelcolor="#e6edf3")

    fig.suptitle(
        "Chinchilla Scaling Laws — Hoffmann et al. 2022", color="#e6edf3", fontsize=13
    )
    plt.tight_layout()
    plt.savefig(
        "chinchilla_scaling.png", dpi=130, bbox_inches="tight", facecolor="#0d1117"
    )
    plt.close()
    print("Saved: chinchilla_scaling.png")


plot_isoflops_curves()

try:
    from IPython.display import Image, display

    display(Image("chinchilla_scaling.png"))
except Exception:
    pass

## 16.1 GPT-3 vs Chinchilla — Was GPT-3 Undertrained?

Using the Chinchilla formula we can compute the **predicted loss** for GPT-3's
actual allocation (175B params, 300B tokens) and compare it to Chinchilla-optimal.


In [ ]:
def loss_scaling_comparison() -> None:
    """Compare actual vs Chinchilla-optimal allocations for landmark models."""
    models = {
        "GPT-3 (175B, 300B tok)": (175e9, 300e9),
        "Chinchilla (70B, 1.4T tok)": (70e9, 1_400e9),
        "LLaMA-1 (65B, 1.4T tok)": (65e9, 1_400e9),
        "LLaMA-2 (70B, 2T tok)": (70e9, 2_000e9),
        "Mistral-7B (7B, ~1T tok est.)": (7e9, 1_000e9),
    }

    print(f"{'Model':<40} {'N':>12} {'D':>14} {'Loss':>8} {'D/N ratio':>12}")
    print("─" * 92)
    for name, (N, D) in models.items():
        loss = chinchilla_loss(N, D)
        ratio = D / N

        # Chinchilla-optimal for same compute
        C = 6 * N * D
        N_opt, D_opt = chinchilla_optimal(C)
        loss_opt = chinchilla_loss(N_opt, D_opt)
        delta = loss - loss_opt

        print(
            f"{name:<40} {N:>12.2e} {D:>14.2e} {loss:>8.4f} "
            f"{ratio:>10.1f}  Δ={delta:+.4f}"
        )

    print("\nΔ = predicted loss above Chinchilla-optimal for same compute.")
    print("Positive Δ → model is undertrained or model is too large.")


loss_scaling_comparison()

# 17) Summary & Key Takeaways

## GPU Architecture & CUDA
- GPUs execute **warps** (32 threads) in SIMT fashion; divergence within a warp
  serialises execution — avoid branch divergence in hot kernels.
- Memory access patterns matter enormously: **coalesced** global reads achieve
  peak HBM bandwidth; strided access can be 10–20× slower.
- The **roofline model** tells you whether your kernel is compute- or
  memory-bandwidth bound, guiding optimisation effort.

## Tensor Cores & Mixed Precision
- Tensor Cores perform **D = A·B + C** in one clock cycle on 4×4 tiles;
  they deliver peak throughput only with aligned, mixed-precision inputs.
- **BF16** is preferred over FP16 for training (same exponent range as FP32);
  **FP8** (H100+) cuts memory another 2× but requires careful amax calibration.
- Always use `torch.amp.autocast` + a GradScaler (or BF16 without scaling).

## Custom Kernels & Profiling
- **Triton** lets you write CUDA-level kernels in Python with ~90 % of hand-
  tuned CUDA performance; ideal for fused ops (softmax, layer norm).
- Use `torch.profiler` to find the actual bottleneck before optimising.

## Parallelism Strategies
| Strategy      | Shards         | Comm per step | Best for |
|---------------|----------------|---------------|----------|
| DDP           | nothing        | all-reduce    | most models |
| Tensor Par.   | weight matrices | all-reduce    | large FFN/attn |
| Pipeline Par. | layers         | P2P           | very deep models |
| ZeRO-3 / FSDP| params+grads+opt| all-gather + reduce-scatter | giant models |

## Scaling Laws
- **Chinchilla**: for a compute budget *C*, optimal N ∝ C^0.5 and D ∝ C^0.5.
- Most pre-2022 models (GPT-3, PaLM) were **over-parameterised relative to
  training data**; modern practice trains smaller models on far more tokens.
- Predicted loss: `L(N, D) = E + A/N^α + B/D^β`; minimise jointly.

---

> "The bitterness of poor GPU utilisation remains long after the sweetness of  
>  fast iteration is forgotten." — distributed ML proverb
